# Prototipo del data agent: registro del lavoro

> **Nota importante.** Questo notebook è il **registro del prototipo**, non l'implementazione di riferimento. Quella si trova in `cleaning.py` (pulizia dei dati), `agent.py` (agente ReAct e tool `python_repl`) e `main.py` (microservizio FastAPI).
>
> Le celle delle sezioni 1-7 mostrano i **numeri dell'epoca**: 1195 righe dopo la deduplica sul file grezzo e 74 grafie di club non ancora normalizzate. Definiscono inoltre funzioni con gli stessi nomi dei moduli (`clean_scouting_data`, `make_python_tool`, `build_data_agent`) nella loro **prima versione**, con firme e comportamenti poi superati.
>
> I numeri e il comportamento attuali sono nella sezione finale **"8. Revisione post-prototipo"**, che importa i moduli veri.

## 1. Setup e caricamento dati

In [1]:
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()
print("API key caricata:", bool(os.getenv("OPENAI_API_KEY")))

df = pd.read_csv("../data/scouting_data.csv")
print(df.shape)
df.head()

API key caricata: True
(1208, 35)


,player_id,player_name,club,nationality,date_of_birth,position,preferred_foot,height_cm,weight_kg,season,...,sprint_speed_kmh,saves,save_pct,clean_sheets,market_value_eur,contract_expiry,scout_rating,injury_days_last_season,observation_date,scout_name
0,13,Delfina Abril Peralta,Vallata Calcio,Argentina,1992-02-03,CM,destro,178,76.7,2024/25,...,31.39,NaN,NaN,NaN,6439000,2027-06-30,55,3.0,2024-08-27,Valeria Costa
1,84,Marcel Guillon du Étienne,Monte Grigio SC,france,1998-09-05,CAM,destro,187,87.4,2025/26,...,32.83,NaN,NaN,NaN,10269000,2026-06-30,62,34.0,24/11/2025,Luca Bianchi
2,36,Marcel Leblanc,Rocca Sportiva,Morocco,1998-12-16,ST,sinistro,184,85.2,2024/25,...,30.26,NaN,NaN,NaN,13929000,2025-12-30,71,0.0,28/09/2024,Elena Conti
3,14,Albano Llopis Hierro,Foresta Nera AC,Colombia,1998-08-30,ST,destro,173,78.3,2024/25,...,31.06,NaN,NaN,NaN,"11.415.000,00",2028-06-30,57,6.0,2024-10-04,Davide Russo
4,72,Ema Marinković,Ponte Vecchio FC,Serbia,2006-06-17,RM,sinistro,170,77.1,2024/25,...,30.91,NaN,NaN,NaN,6172000,2028-06-30,63,9.0,2025-04-25,Giulia Ricci


## 2. Esplorazione iniziale

In [2]:
df.info()
print("\n--- Valori mancanti per colonna ---")
print(df.isna().sum())
print("\n--- Duplicati (righe identiche) ---")
print(df.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 1208 entries, 0 to 1207
Data columns (total 35 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   player_id                1208 non-null   int64  
 1   player_name              1208 non-null   str    
 2   club                     1208 non-null   str    
 3   nationality              1208 non-null   str    
 4   date_of_birth            1208 non-null   str    
 5   position                 1208 non-null   str    
 6   preferred_foot           1208 non-null   str    
 7   height_cm                1208 non-null   int64  
 8   weight_kg                1185 non-null   float64
 9   season                   1208 non-null   str    
 10  matches_played           1208 non-null   int64  
 11  minutes_played           1208 non-null   int64  
 12  goals                    1208 non-null   int64  
 13  assists                  1208 non-null   int64  
 14  xg                       1149 non-n

## 3. Analisi anomalie specifiche

In [3]:
print("--- Valori unici in nationality ---")
print(sorted(df['nationality'].unique()))

print("\n--- Esempi di market_value_eur (valori non puramente numerici) ---")
print(df[df['market_value_eur'].str.contains(r'[^\d]', regex=True)]['market_value_eur'].unique())

print("\n--- Esempi di formati in observation_date ---")
print(df['observation_date'].sample(10, random_state=1).tolist())

print("\n--- Esempi di formati in contract_expiry ---")
print(df['contract_expiry'].sample(10, random_state=1).tolist())

print("\n--- player_name con caratteri non latini ---")
print(df[df['player_name'].str.contains(r'[^\x00-\x7F]', regex=True)]['player_name'].unique())

--- Valori unici in nationality ---
['Argentina', 'Austria', 'Belgium', 'Brazil', 'Colombia', 'Croatia', 'Denmark', 'England', 'France', 'Germany', 'Ghana', 'Italy', 'Ivory Coast', 'Japan', 'Morocco', 'Netherlands', 'Nigeria', 'Norway', 'Poland', 'Senegal', 'Serbia', 'Spain', 'Turkey', 'Uruguay', 'argentina', 'austria', 'belgium', 'brazil', 'colombia', 'croatia', 'denmark', 'england', 'france', 'germany', 'italy', 'ivory coast', 'japan', 'morocco', 'netherlands', 'norway', 'poland', 'senegal', 'spain', 'turkey', 'uruguay']

--- Esempi di market_value_eur (valori non puramente numerici) ---
<StringArray>
['11.415.000,00',  '7.311.000,00',  '6.635.000,00',  '7.248.000,00',
  '1.696.000,00',  '4.531.000,00',  '8.499.000,00',  '4.086.000,00',
  '4.450.000,00',  '4.624.000,00',
 ...
  '7.668.000,00',  '6.185.000,00',  '5.617.000,00',  '3.730.000,00',
  '3.906.000,00',  '5.429.000,00',  '8.824.000,00', '19.102.000,00',
  '8.455.000,00',  '5.767.000,00']
Length: 105, dtype: str

--- Esempi di

## 4. Cleaning

Applichiamo la procedura di data cleaning descritta nella sezione 6 di `platform_handbook.md`, un passaggio alla volta, verificando il risultato dopo ogni correzione.


### 4.1 Rimozione duplicati e normalizzazione `nationality`

In [4]:
df_clean = df.drop_duplicates().copy()
print("Righe dopo rimozione duplicati:", len(df_clean))

df_clean['nationality'] = df_clean['nationality'].str.strip().str.title()
print(sorted(df_clean['nationality'].unique()))

Righe dopo rimozione duplicati: 1195
['Argentina', 'Austria', 'Belgium', 'Brazil', 'Colombia', 'Croatia', 'Denmark', 'England', 'France', 'Germany', 'Ghana', 'Italy', 'Ivory Coast', 'Japan', 'Morocco', 'Netherlands', 'Nigeria', 'Norway', 'Poland', 'Senegal', 'Serbia', 'Spain', 'Turkey', 'Uruguay']


### 4.2 Normalizzazione `market_value_eur`

Il campo contiene due formati misti: numeri puri (es. `6439000`) e formato italiano con punti/virgola (es. `11.415.000,00`). Li convertiamo entrambi in float.

In [5]:
def parse_market_value(value):
    value = str(value).strip()
    if ',' in value and '.' in value:
        # formato italiano: punti = migliaia, virgola = decimali
        value = value.replace('.', '').replace(',', '.')
    return float(value)

df_clean['market_value_eur'] = df_clean['market_value_eur'].apply(parse_market_value)
print(df_clean['market_value_eur'].describe())
print(df_clean['market_value_eur'].head(10))

count    1.195000e+03
mean     7.161197e+06
std      2.823789e+06
min      1.618000e+06
25%      5.041000e+06
50%      6.959000e+06
75%      8.659000e+06
max      1.910200e+07
Name: market_value_eur, dtype: float64
0     6439000.0
1    10269000.0
2    13929000.0
3    11415000.0
4     6172000.0
5     9402000.0
6     9741000.0
7     5253000.0
8     5332000.0
9     8473000.0
Name: market_value_eur, dtype: float64


### 4.3 Normalizzazione date (`observation_date`, `contract_expiry`)

Le date sono registrate in due formati diversi (`AAAA-MM-GG` e `GG/MM/AAAA`), a seconda del canale di raccolta. Le uniformiamo a `datetime64`.

In [6]:
def parse_mixed_date(value):
    value = str(value).strip()
    # prova prima il formato ISO, poi quello GG/MM/AAAA
    try:
        return pd.to_datetime(value, format='%Y-%m-%d')
    except ValueError:
        return pd.to_datetime(value, format='%d/%m/%Y')

df_clean['observation_date'] = df_clean['observation_date'].apply(parse_mixed_date)
df_clean['contract_expiry'] = df_clean['contract_expiry'].apply(parse_mixed_date)

print(df_clean['observation_date'].dtype)
print(df_clean['observation_date'].head(10))
print(df_clean['observation_date'].min(), "-", df_clean['observation_date'].max())

datetime64[us]
0   2024-08-27
1   2025-11-24
2   2024-09-28
3   2024-10-04
4   2025-04-25
5   2024-08-23
6   2025-01-07
7   2024-03-14
8   2024-11-18
9   2023-11-09
Name: observation_date, dtype: datetime64[us]
2023-08-16 00:00:00 - 2026-05-20 00:00:00


### 4.4 Controllo range e individuazione anomalie

Verifichiamo che le metriche percentuali stiano in [0,100] e che i valori fisici (velocità, altezza) siano plausibili, come richiesto dalla sezione 6 punto 5 dell'handbook.

In [7]:
print("Range contract_expiry:", df_clean['contract_expiry'].min(), "-", df_clean['contract_expiry'].max())

pct_cols = ['pass_accuracy_pct', 'duels_won_pct', 'aerial_duels_won_pct', 'save_pct']
for col in pct_cols:
    fuori_range = df_clean[(df_clean[col] < 0) | (df_clean[col] > 100)]
    print(f"{col}: fuori range [0,100] -> {len(fuori_range)} righe")

print("\nsprint_speed_kmh describe:")
print(df_clean['sprint_speed_kmh'].describe())

print("\nheight_cm describe:")
print(df_clean['height_cm'].describe())

Range contract_expiry: 2024-06-30 00:00:00 - 2029-12-30 00:00:00
pass_accuracy_pct: fuori range [0,100] -> 18 righe
duels_won_pct: fuori range [0,100] -> 0 righe
aerial_duels_won_pct: fuori range [0,100] -> 0 righe
save_pct: fuori range [0,100] -> 0 righe

sprint_speed_kmh describe:
count    1149.000000
mean       30.453281
std         6.763150
min       -36.330000
25%        29.610000
50%        30.910000
75%        32.660000
max        37.500000
Name: sprint_speed_kmh, dtype: float64

height_cm describe:
count    1195.000000
mean      197.492887
std       164.487124
min       163.000000
25%       176.000000
50%       181.000000
75%       185.000000
max      1960.000000
Name: height_cm, dtype: float64


In [8]:
print("--- pass_accuracy_pct fuori range ---")
print(df_clean[(df_clean['pass_accuracy_pct'] < 0) | (df_clean['pass_accuracy_pct'] > 100)]['pass_accuracy_pct'].tolist())

print("\n--- sprint_speed_kmh negativi ---")
print(df_clean[df_clean['sprint_speed_kmh'] < 0]['sprint_speed_kmh'].tolist())

print("\n--- height_cm anomali (fuori 150-210) ---")
print(df_clean[(df_clean['height_cm'] < 150) | (df_clean['height_cm'] > 210)][['player_id', 'player_name', 'height_cm']])

--- pass_accuracy_pct fuori range ---
[183.2, 175.7, 180.0, 182.2, 180.0, 167.0, 180.0, 186.2, 183.0, 180.5, 191.2, 187.1, 172.5, 177.1, 175.9, 177.9, 167.5, 160.1]

--- sprint_speed_kmh negativi ---
[-30.12, -29.96, -33.31, -30.0, -28.61, -36.33, -26.77, -33.69, -33.99, -27.68, -30.79, -33.57]

--- height_cm anomali (fuori 150-210) ---
      player_id                     player_name  height_cm
63           34           Conrado Rivas-Giménez       1810
112         130            Gabriel Farias Lopez       1850
120          60                    Hanna Nygård       1890
157         127            Herwig Hering-Bruder       1960
405          12            Helge Ahmed-Svendsen       1820
411          49                  Janos Schleich       1840
619         104                Augustin Foucher       1850
741          89                    Zachary Rice       1820
797          67       Charles-Laurent Chevalier       1740
1012         68         Georg Trattner-Reiterer       1750
1037        

### 4.5 Correzione anomalie

Le anomalie trovate seguono pattern sistematici e non casuali: `pass_accuracy_pct` fuori range è sempre il doppio del valore plausibile, `sprint_speed_kmh` negativo ha il segno invertito, `height_cm` anomalo è 10 volte il valore corretto. Correggiamo di conseguenza invece di scartare le righe.

In [9]:
df_clean.loc[df_clean['pass_accuracy_pct'] > 100, 'pass_accuracy_pct'] /= 2
df_clean['sprint_speed_kmh'] = df_clean['sprint_speed_kmh'].abs()
df_clean.loc[df_clean['height_cm'] > 210, 'height_cm'] /= 10

# verifica
print("pass_accuracy_pct fuori range:", ((df_clean['pass_accuracy_pct'] < 0) | (df_clean['pass_accuracy_pct'] > 100)).sum())
print("sprint_speed_kmh negativi:", (df_clean['sprint_speed_kmh'] < 0).sum())
print("height_cm fuori [150,210]:", ((df_clean['height_cm'] < 150) | (df_clean['height_cm'] > 210)).sum())

pass_accuracy_pct fuori range: 0
sprint_speed_kmh negativi: 0
height_cm fuori [150,210]: 0


### 4.6 Imputazione valori mancanti

Seguendo la procedura handbook, imputiamo i valori mancanti con la mediana calcolata per combinazione ruolo/stagione, evitando un drop indiscriminato delle righe. Le colonne `saves`/`save_pct`/`clean_sheets` sono escluse: i loro mancanti sono normali (vuoti per costruzione per i non portieri), non vanno imputati.

In [10]:
cols_da_imputare = ['weight_kg', 'xg', 'xa', 'pass_accuracy_pct', 'sprint_speed_kmh', 'injury_days_last_season']

for col in cols_da_imputare:
    prima = df_clean[col].isna().sum()
    df_clean[col] = df_clean.groupby(['position', 'season'])[col].transform(
        lambda x: x.fillna(x.median())
    )
    dopo = df_clean[col].isna().sum()
    print(f"{col}: {prima} mancanti -> {dopo} mancanti dopo imputazione")

weight_kg: 23 mancanti -> 0 mancanti dopo imputazione
xg: 59 mancanti -> 0 mancanti dopo imputazione
xa: 60 mancanti -> 0 mancanti dopo imputazione
pass_accuracy_pct: 34 mancanti -> 0 mancanti dopo imputazione
sprint_speed_kmh: 46 mancanti -> 0 mancanti dopo imputazione
injury_days_last_season: 73 mancanti -> 0 mancanti dopo imputazione


### 4.7 Traslitterazione nomi non latini

Un sottoinsieme di `player_name` contiene caratteri kanji giapponesi. Proviamo prima con la libreria `unidecode`.

In [11]:
from unidecode import unidecode

mask_non_latino = df_clean['player_name'].str.contains(r'[^\x00-\x7F]', regex=True) & df_clean['player_name'].str.contains(r'[\u3040-\u30ff\u4e00-\u9fff]', regex=True)

print("Nomi da traslitterare:", mask_non_latino.sum())
print(df_clean.loc[mask_non_latino, 'player_name'].tolist())

df_clean.loc[mask_non_latino, 'player_name'] = df_clean.loc[mask_non_latino, 'player_name'].apply(unidecode)

print("\nDopo traslitterazione:")
print(df_clean.loc[mask_non_latino, 'player_name'].tolist())

Nomi da traslitterare: 79
['伊藤 裕太', '池田 英樹', '伊藤 裕太', '吉田 智也', '吉田 直人', '松本 学', '吉田 直人', '松本 学', '佐藤 淳', '吉田 智也', '林 拓真', '前田 健一', '佐藤 淳', '吉田 直人', '前田 健一', '吉田 智也', '吉田 智也', '伊藤 裕太', '吉田 智也', '松本 学', '佐藤 裕太', '池田 英樹', '佐藤 裕太', '佐藤 淳', '松本 学', '吉田 智也', '池田 英樹', '佐藤 裕太', '吉田 智也', '池田 英樹', '前田 健一', '林 拓真', '松本 学', '伊藤 裕太', '松本 学', '前田 健一', '池田 英樹', '松本 学', '伊藤 裕太', '池田 英樹', '佐藤 裕太', '林 拓真', '林 拓真', '林 拓真', '前田 健一', '伊藤 裕太', '吉田 智也', '前田 健一', '佐藤 淳', '池田 英樹', '佐藤 淳', '林 拓真', '伊藤 裕太', '吉田 直人', '林 拓真', '佐藤 淳', '前田 健一', '佐藤 淳', '佐藤 裕太', '松本 学', '吉田 直人', '池田 英樹', '吉田 直人', '松本 学', '前田 健一', '佐藤 淳', '吉田 直人', '前田 健一', '吉田 智也', '佐藤 裕太', '林 拓真', '佐藤 裕太', '林 拓真', '佐藤 淳', '林 拓真', '佐藤 淳', '吉田 智也', '前田 健一', '前田 健一']

Dopo traslitterazione:
['Yi Teng  Yu Tai ', 'Chi Tian  Ying Shu ', 'Yi Teng  Yu Tai ', 'Ji Tian  Zhi Ye ', 'Ji Tian  Zhi Ren ', 'Song Ben  Xue ', 'Ji Tian  Zhi Ren ', 'Song Ben  Xue ', 'Zuo Teng  Chun ', 'Ji Tian  Zhi Ye ', 'Lin  Tuo Zhen ', 'Qian Tian  Jian Yi ', 'Zuo Teng  Chun ', 'Ji Ti

⚠️ `unidecode` traslittera i kanji con lettura **cinese** (pinyin) e non giapponese, producendo risultati foneticamente sbagliati (es. `伊藤 裕太` → `Yi Teng Yu Tai` invece di `Ito Yuta`). Correggiamo con una mappatura esplicita delle letture giapponesi corrette (Hepburn).

In [12]:
mappa_correzione = {
    'Yi Teng  Yu Tai ': 'Ito Yuta',
    'Chi Tian  Ying Shu ': 'Ikeda Hideki',
    'Ji Tian  Zhi Ye ': 'Yoshida Tomoya',
    'Ji Tian  Zhi Ren ': 'Yoshida Naoto',
    'Song Ben  Xue ': 'Matsumoto Manabu',
    'Zuo Teng  Chun ': 'Sato Jun',
    'Lin  Tuo Zhen ': 'Hayashi Takuma',
    'Qian Tian  Jian Yi ': 'Maeda Kenichi',
    'Zuo Teng  Yu Tai ': 'Sato Yuta',
}

df_clean['player_name'] = df_clean['player_name'].replace(mappa_correzione)

print(df_clean.loc[mask_non_latino, 'player_name'].unique())

<StringArray>
[        'Ito Yuta',     'Ikeda Hideki',   'Yoshida Tomoya',
    'Yoshida Naoto', 'Matsumoto Manabu',         'Sato Jun',
   'Hayashi Takuma',    'Maeda Kenichi',        'Sato Yuta']
Length: 9, dtype: str


## 5. Funzione di cleaning consolidata

Raccogliamo tutti i passaggi precedenti in un'unica funzione riutilizzabile, che richiameremo anche dal futuro microservizio FastAPI. Verifichiamo che, applicata da zero al dataset originale, produca lo stesso risultato ottenuto manualmente.

In [13]:
def clean_scouting_data(df_raw):
    df = df_raw.drop_duplicates().copy()

    # normalizzazione nationality
    df['nationality'] = df['nationality'].str.strip().str.title()

    # market_value_eur: stringa mista -> float
    def parse_market_value(value):
        value = str(value).strip()
        if ',' in value and '.' in value:
            value = value.replace('.', '').replace(',', '.')
        return float(value)
    df['market_value_eur'] = df['market_value_eur'].apply(parse_market_value)

    # date miste -> datetime
    def parse_mixed_date(value):
        value = str(value).strip()
        try:
            return pd.to_datetime(value, format='%Y-%m-%d')
        except ValueError:
            return pd.to_datetime(value, format='%d/%m/%Y')
    df['observation_date'] = df['observation_date'].apply(parse_mixed_date)
    df['contract_expiry'] = df['contract_expiry'].apply(parse_mixed_date)

    # correzione anomalie di range
    df.loc[df['pass_accuracy_pct'] > 100, 'pass_accuracy_pct'] /= 2
    df['sprint_speed_kmh'] = df['sprint_speed_kmh'].abs()
    df.loc[df['height_cm'] > 210, 'height_cm'] /= 10

    # imputazione valori mancanti (mediana per ruolo/stagione)
    cols_da_imputare = ['weight_kg', 'xg', 'xa', 'pass_accuracy_pct', 'sprint_speed_kmh', 'injury_days_last_season']
    for col in cols_da_imputare:
        df[col] = df.groupby(['position', 'season'])[col].transform(lambda x: x.fillna(x.median()))

    # traslitterazione nomi giapponesi
    mappa_correzione_nomi = {
        '伊藤 裕太': 'Ito Yuta',
        '池田 英樹': 'Ikeda Hideki',
        '吉田 智也': 'Yoshida Tomoya',
        '吉田 直人': 'Yoshida Naoto',
        '松本 学': 'Matsumoto Manabu',
        '佐藤 淳': 'Sato Jun',
        '林 拓真': 'Hayashi Takuma',
        '前田 健一': 'Maeda Kenichi',
        '佐藤 裕太': 'Sato Yuta',
    }
    df['player_name'] = df['player_name'].replace(mappa_correzione_nomi)

    return df

# test: rilanciamo la funzione da zero sul CSV originale e confrontiamo col df_clean fatto a mano
df_test = clean_scouting_data(df)
print("Shape:", df_test.shape)
print("Uguale a df_clean?", df_test.equals(df_clean))

Shape: (1195, 35)
Uguale a df_clean? True


## 6. Agente ReAct per l'analisi dati

Il brief richiede un "pandas agent" per l'analisi del CSV. Abbiamo scelto di non usare
`create_pandas_dataframe_agent` di `langchain_experimental`, perché la libreria è deprecata
("in sunset", vedi warning ufficiale) e la sua API non è pensata per la versione 1.x di LangChain
che usiamo in questo progetto.

Costruiamo invece un agente ReAct generico (`create_agent` di `langchain.agents`) dotato di un
unico tool custom, `python_repl`, che esegue codice Python arbitrario con accesso al dataframe.
Questo approccio è più flessibile del pandas agent di libreria (permette anche di generare grafici,
non solo query tabellari) e resta comunque un "interprete deterministico di codice che analizza il
CSV" — lo stesso concetto richiesto dal brief, con un'implementazione più moderna e sotto il nostro
controllo diretto (possiamo ispezionare e correggere ogni bug, cosa che sarebbe stata più difficile
con una libreria di terze parti in via di dismissione).

In [14]:
# Import mantenuto solo a scopo dimostrativo: la libreria langchain_experimental segnala
# esplicitamente questa funzione come deprecata/in sunset (vedi DeprecationWarning sotto).
# Per questo motivo NON la usiamo per costruire l'agente (vedi motivazione nella cella markdown
# sopra) — costruiamo invece un agente ReAct con tool custom nelle celle successive.
from langchain_experimental.agents import create_pandas_dataframe_agent
print("Import riuscito (solo a scopo dimostrativo, non usato oltre)")

/var/folders/67/6cdkb9zj3172d625s_6c70x00000gn/T/ipykernel_18577/2430767143.py:5: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.agents import create_pandas_dataframe_agent


Import riuscito (solo a scopo dimostrativo, non usato oltre)


### 6.1 Tool di esecuzione codice Python

Il cuore dell'agente è un singolo tool, `python_repl`, che espone al modello linguistico
la possibilità di eseguire codice Python arbitrario contro il dataframe `df_clean`.

Caratteristiche costruite/corrette durante i test (vedi sezione 6.3 per il dettaglio dei bug osservati):
- **Comportamento REPL** (via `ast.parse`): se l'ultima istruzione del codice generato dall'LLM
  è un'espressione (es. `df.head()`), il suo valore viene mostrato automaticamente, come in una
  cella Jupyter — senza questo, l'LLM doveva ricordarsi di chiamare sempre `print()` esplicitamente,
  e spesso se ne dimenticava.
- **Stato persistente tra chiamate**: `local_vars` è creato una sola volta quando si costruisce il
  tool (non ad ogni chiamata di `python_repl`), quindi le variabili definite in una tool-call restano
  disponibili nelle chiamate successive, anche tra invocazioni separate dell'agente.
- **Generazione grafici**: se il codice genera una figura matplotlib, viene rilevata e salvata
  automaticamente come PNG in `charts/`, e il tool comunica il percorso esatto all'LLM nel testo
  di ritorno (per evitare che il modello dichiari un "successo" non verificato).
- **Gestione errori**: ogni chiamata gira in un blocco try/except che restituisce l'errore come
  stringa leggibile, permettendo il ciclo di auto-correzione tipico del pattern ReAct.

In [15]:
import io
import contextlib
import ast
import uuid
import os
import matplotlib
matplotlib.use("Agg")  # backend non interattivo: necessario per salvare figure senza un display
import matplotlib.pyplot as plt
import seaborn as sns
from langchain_core.tools import tool

CHARTS_DIR = "charts"
os.makedirs(CHARTS_DIR, exist_ok=True)


def make_python_tool(df):
    # Stato persistente: 'local_vars' viene creato UNA VOLTA qui (non dentro python_repl),
    # quindi è condiviso tra tutte le chiamate al tool, anche in invocazioni separate dell'agente.
    local_vars = {"df": df, "pd": pd, "plt": plt, "sns": sns}

    @tool
    def python_repl(code: str) -> str:
        """Esegue codice Python per analizzare il dataframe (disponibile come variabile 'df').
        Disponibili anche 'pd', 'plt' (matplotlib.pyplot) e 'sns' (seaborn) per creare grafici:
        se il codice genera una figura matplotlib, viene salvata automaticamente come PNG
        (non serve chiamare plt.savefig() né plt.show()).
        Le variabili definite in una chiamata restano disponibili nelle chiamate successive
        (come in un notebook Jupyter).
        Si comporta come un REPL: se l'ultima riga è un'espressione, il suo valore viene mostrato
        automaticamente, altrimenti usa print()."""
        output = io.StringIO()
        chart_note = ""
        try:
            # Parsiamo il codice per distinguere l'ultima istruzione dal resto: se è
            # un'espressione nuda (es. 'df.head()'), la valutiamo e ne stampiamo il repr,
            # replicando il comportamento "auto-print" di una cella Jupyter.
            tree = ast.parse(code)
            *body, last = tree.body if tree.body else [None]

            with contextlib.redirect_stdout(output):
                if body:
                    exec(compile(ast.Module(body=body, type_ignores=[]), "<code>", "exec"), {}, local_vars)
                if last is not None:
                    if isinstance(last, ast.Expr):
                        result = eval(compile(ast.Expression(last.value), "<code>", "eval"), {}, local_vars)
                        if result is not None:
                            print(repr(result))
                    else:
                        exec(compile(ast.Module(body=[last], type_ignores=[]), "<code>", "exec"), {}, local_vars)

                # Se il codice ha creato una figura matplotlib, la salviamo e lo comunichiamo
                # esplicitamente all'LLM (dentro il blocco catturato, cosi il messaggio finisce
                # davvero nel valore di ritorno del tool e non si perde nello stdout reale).
                if plt.get_fignums():
                    filename = f"{CHARTS_DIR}/chart_{uuid.uuid4().hex[:8]}.png"
                    plt.savefig(filename, bbox_inches="tight")
                    plt.close("all")
                    chart_note = f"\n[Grafico salvato correttamente in: {filename}]"
        except Exception as e:
            plt.close("all")
            return f"Errore nell'esecuzione: {e}"
        return (output.getvalue() or "Codice eseguito senza output.") + chart_note
    return python_repl

### 6.2 Costruzione dell'agente ReAct

Costruiamo l'agente con `create_agent` (da `langchain.agents`, LangChain 1.x — non il vecchio
`create_react_agent` di `langgraph.prebuilt`). Prima di usarlo abbiamo verificato la firma esatta
con `help(create_agent)`: il parametro per il messaggio di sistema si chiama **`system_prompt`**
(non `prompt`, come si potrebbe intuire da tutorial più datati).

Il `system_prompt` non contiene solo lo schema del dataframe: include anche una serie di regole
di business aggiunte incrementalmente in risposta a bug osservati durante i test (dettagliati
nella sezione 6.3), in particolare sulla granularità temporale dei dati e sulla necessità di
verificare i valori delle colonne categoriali prima di filtrare.

In [16]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print(llm)

metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2', 'langchain-openai': '1.6.6'}} profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True} client=<openai.resources.chat.completions.completions.Completions object at 0x147585550> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x147585fd0> root_client=<openai.OpenAI object at 0x147584d70> root_async_client=<openai.AsyncOpen

In [ ]:
from langchain.agents import create_agent

help(create_agent)

In [18]:
system_prompt_text = f"""Colonne disponibili nel dataframe 'df': {list(df_clean.columns)}

Nota importante sulla struttura dei dati: ogni riga rappresenta una singola osservazione di scouting
in un momento nel tempo (colonna 'observation_date'). Lo stesso giocatore ('player_id') compare più
volte, e metriche come 'market_value_eur' o 'scout_rating' possono cambiare da un'osservazione
all'altra nel tempo.

Usa sempre 'player_id' come chiave univoca del giocatore (mai 'player_name', per rischio di omonimie).

Se la domanda riguarda lo stato ATTUALE di un giocatore (es. valore di mercato attuale, ultimo rating),
NON prendere il valore massimo/minimo storico. Usa esattamente questo procedimento per ottenere
la riga più recente per ciascun giocatore:
    df.sort_values('observation_date').drop_duplicates(subset='player_id', keep='last')
Poi applica eventuali filtri o ordinamenti richiesti dalla domanda su questo sottoinsieme.

Se una domanda richiede informazioni non presenti tra le colonne del dataframe, NON rispondere con
conoscenza generale esterna: dichiara esplicitamente che il dataset non contiene quell'informazione,
elencando le colonne disponibili più pertinenti alla domanda.

Prima di filtrare su una colonna categoriale (es. 'position', 'preferred_foot', 'nationality') con un
valore specifico di cui non sei certo, controlla SEMPRE prima i valori validi con df['colonna'].unique(),
invece di assumere un valore plausibile. Se un filtro produce un risultato vuoto, non concludere che il
dato non esiste nel dataset: verifica prima se stai usando il valore/etichetta corretta."""

python_tool = make_python_tool(df_clean)
agent = create_agent(llm, [python_tool], system_prompt=system_prompt_text)

### 6.3 Test dell'agente su casi limite

Prima di consolidare tutto in una funzione riutilizzabile (sezione 7), abbiamo verificato il
comportamento dell'agente su una serie di domande via via più impegnative, correggendo il
`system_prompt` e il tool ogni volta che un test rivelava un problema. Questo percorso di
bug-fix è di per sé un buon esempio di "sfida tecnica" da citare in presentazione.

Riepilogo dei test e di cosa hanno rivelato:
1. **Aggregazione semplice** (media per ruolo) — funziona subito.
2. **Top N con filtro + ordinamento** — ha rivelato il problema della granularità temporale dei
   dati (lo stesso giocatore compare più volte nel tempo, con valori diversi) e la differenza tra
   "valore massimo storico" e "valore più recente"; il `system_prompt` è stato rafforzato con la
   procedura esatta da seguire (non bastava descriverla a parole).
3. **Generazione di un grafico** — ha rivelato un bug nel tool: il messaggio di conferma del
   salvataggio finiva fuori dal buffer catturato e non arrivava mai all'LLM, che quindi dichiarava
   "successo" senza un riscontro reale (poi corretto).
4. **Domanda fuori dallo scope del dataset** (stipendio degli scout) — verifica che l'agente non
   risponda con conoscenza generale esterna ma dichiari esplicitamente il limite del dataset.
5. **Colonna con missing-by-design** (percentuale parate dei portieri) — ha rivelato che l'LLM può
   assumere un valore categoriale plausibile ma sbagliato (es. `'Goalkeeper'` invece di `'GK'`) e
   trarre una conclusione errata da un filtro vuoto; corretto imponendo la verifica di `.unique()`
   prima di ogni filtro categoriale incerto.
6. **Stato persistente tra chiamate separate** — verifica esplicita (con una variabile "sentinella")
   che l'ambiente Python condiviso dal tool ricordi le variabili anche tra `agent.invoke()` distinti,
   confermando che il fix allo stato persistente funziona davvero.

In [19]:
risposta1 = agent.invoke({
    "messages": [{"role": "user", "content": "Qual è lo scout_rating medio per ruolo? Rispondi in modo discorsivo."}]
})

print(risposta1["messages"][-1].content)

Analizzando lo scout_rating medio per ruolo, emergono alcune interessanti osservazioni. I giocatori che ricoprono il ruolo di **Centrocampista Offensivo (CAM)** hanno il punteggio medio più alto, con un valore di **69.36**. Questo suggerisce che i CAM tendono ad avere un potenziale di scouting superiore rispetto ad altri ruoli.

Al secondo posto troviamo i **Portieri (GK)**, con uno scout_rating medio di **68.29**, il che indica che i portieri di questo dataset sono considerati di buon livello. 

I **Centrocampisti (CM)** e i **Centrocampisti Difensivi (CDM)** seguono, con punteggi medi rispettivamente di **65.40** e **63.70**. Questi valori evidenziano l'importanza dei centrocampisti nel contesto del gioco, sia in fase offensiva che difensiva.

Per quanto riguarda i difensori, i **Difensori Centrali (CB)** hanno uno scout_rating medio di **62.17**, mentre i **Difensori Laterali (LB e RB)** si attestano su valori di **65.53** e **59.67**, rispettivamente. Questo potrebbe indicare una l

In [20]:
for msg in risposta1["messages"]:
    print(f"--- {msg.type} ---")
    print(msg.content)
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print("Tool calls:", msg.tool_calls)
    print()

--- human ---
Qual è lo scout_rating medio per ruolo? Rispondi in modo discorsivo.

--- ai ---

Tool calls: [{'name': 'python_repl', 'args': {'code': "# Prima di calcolare lo scout_rating medio per ruolo, dobbiamo ottenere la riga più recente per ciascun giocatore.\n# Ordiniamo il dataframe per 'observation_date' e manteniamo solo l'ultima osservazione per ogni 'player_id'.\nlatest_observations = df.sort_values('observation_date').drop_duplicates(subset='player_id', keep='last')\n\n# Ora calcoliamo lo scout_rating medio per ruolo.\naverage_scout_rating_by_position = latest_observations.groupby('position')['scout_rating'].mean().reset_index()\naverage_scout_rating_by_position"}, 'id': 'call_XlB6bJxpieD6XS7yidKjGWSX', 'type': 'tool_call'}]

--- tool ---
   position  scout_rating
0       CAM     69.357143
1        CB     62.166667
2       CDM     63.700000
3        CM     65.400000
4        GK     68.285714
5        LB     65.526316
6        LM     65.000000
7        LW     68.444444
8   

**Test 2 — top N con filtro, ordinamento e deduplica temporale.**
Prima versione: mostra il bug della granularità temporale (l'agente confonde valore massimo
storico e valore più recente). Il `system_prompt` viene poi rafforzato di conseguenza.

In [21]:
risposta2 = agent.invoke({
    "messages": [{"role": "user", "content": "Chi sono i 5 giocatori con il market_value_eur più alto nel ruolo ST? Dammi nome e valore."}]
})

print(risposta2["messages"][-1].content)

I 5 giocatori con il market_value_eur più alto nel ruolo di attaccante (ST) sono:

1. Bulunç Müslum Kısakürek Aslan - 13,423,000 EUR
2. Zachary Rice - 10,691,000 EUR
3. Semender Yekda Hayrioğlu Bilir - 10,316,000 EUR
4. Marcel Leblanc - 9,329,000 EUR
5. Erik Cox - 9,272,000 EUR


In [22]:
for msg in risposta2["messages"]:
    print(f"--- {msg.type} ---")
    print(msg.content)
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print("Tool calls:", msg.tool_calls)
    print()

--- human ---
Chi sono i 5 giocatori con il market_value_eur più alto nel ruolo ST? Dammi nome e valore.

--- ai ---

Tool calls: [{'name': 'python_repl', 'args': {'code': "# Filtriamo il dataframe per il ruolo 'ST' e otteniamo l'osservazione più recente per ciascun giocatore\nrecent_players = df.sort_values('observation_date').drop_duplicates(subset='player_id', keep='last')\n\n# Filtriamo per il ruolo 'ST'\nst_players = recent_players[recent_players['position'] == 'ST']\n\n# Ordiniamo per 'market_value_eur' e prendiamo i primi 5\ntop_st_players = st_players.nlargest(5, 'market_value_eur')[['player_name', 'market_value_eur']]\n\ntop_st_players"}, 'id': 'call_7ol3cOMFFO7vH6dgxX9223rr', 'type': 'tool_call'}]

--- tool ---
                         player_name  market_value_eur
729    Bulunç Müslum Kısakürek Aslan        13423000.0
768                     Zachary Rice        10691000.0
835   Semender Yekda Hayrioğlu Bilir        10316000.0
827                   Marcel Leblanc         9329

In [23]:
df_clean[df_clean['player_name'] == 'Erik Cox'][['player_id', 'position', 'season', 'market_value_eur', 'observation_date']]

,player_id,position,season,market_value_eur,observation_date
97,118,ST,2025/26,9272000.0,2025-09-21
225,118,ST,2023/24,16352000.0,2023-12-25
315,118,ST,2023/24,16352000.0,2023-09-20
393,118,ST,2024/25,10438000.0,2024-10-14
522,118,ST,2025/26,9272000.0,2026-02-14
830,118,ST,2025/26,9272000.0,2025-10-08
983,118,ST,2024/25,10438000.0,2025-03-02
1107,118,ST,2025/26,9272000.0,2026-04-29
1127,118,ST,2023/24,16352000.0,2023-12-09


**Test 3 — generazione e salvataggio di un grafico.**
Verifica che il tool rilevi automaticamente una figura matplotlib e la salvi, e che l'LLM riceva
conferma reale del salvataggio (non un'assunzione).

In [24]:
risposta3 = agent.invoke({
    "messages": [{"role": "user", "content": "Crea un grafico a barre della distribuzione dei giocatori per ruolo (position), usando l'osservazione più recente per ciascun giocatore."}]
})

print(risposta3["messages"][-1].content)

<code>:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


Ho creato il grafico a barre della distribuzione dei giocatori per ruolo, utilizzando l'osservazione più recente per ciascun giocatore. Puoi visualizzarlo qui:

![Distribuzione dei Giocatori per Ruolo](sandbox:/mnt/data/distribuzione_giocatori_per_ruolo.png)


In [25]:
for msg in risposta3["messages"]:
    print(f"--- {msg.type} ---")
    print(msg.content)
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print("Tool calls:", msg.tool_calls)
    print()

--- human ---
Crea un grafico a barre della distribuzione dei giocatori per ruolo (position), usando l'osservazione più recente per ciascun giocatore.

--- ai ---

Tool calls: [{'name': 'python_repl', 'args': {'code': "import pandas as pd\nimport matplotlib.pyplot as plt\n\n# Ottieni l'osservazione più recente per ciascun giocatore\nrecent_players = df.sort_values('observation_date').drop_duplicates(subset='player_id', keep='last')\n\n# Conta il numero di giocatori per ruolo\nposition_counts = recent_players['position'].value_counts()\n\n# Crea il grafico a barre\nplt.figure(figsize=(10, 6))\nposition_counts.plot(kind='bar', color='skyblue')\nplt.title('Distribuzione dei Giocatori per Ruolo')\nplt.xlabel('Ruolo')\nplt.ylabel('Numero di Giocatori')\nplt.xticks(rotation=45)\nplt.grid(axis='y')\nplt.tight_layout()\n\n# Salva il grafico come PNG\nplt.savefig('/mnt/data/distribuzione_giocatori_per_ruolo.png')\nplt.show()"}, 'id': 'call_vOuru2klYPFCYDZHlDv5XmyX', 'type': 'tool_call'}]

--- t

**Test 4 — domanda fuori dallo scope del dataset.**
Nessuna colonna sugli stipendi degli scout: verifichiamo che l'agente non "allucini" una risposta
con conoscenza generale, ma dichiari esplicitamente il limite dei dati disponibili.

In [26]:
risposta4 = agent.invoke({
    "messages": [{"role": "user", "content": "Qual è lo stipendio medio degli scout?"}]
})

print(risposta4["messages"][-1].content)

Il dataset non contiene informazioni sugli stipendi degli scout. Le colonne disponibili nel dataframe sono: ['player_id', 'player_name', 'club', 'nationality', 'date_of_birth', 'position', 'preferred_foot', 'height_cm', 'weight_kg', 'season', 'matches_played', 'minutes_played', 'goals', 'assists', 'xg', 'xa', 'pass_accuracy_pct', 'duels_won_pct', 'aerial_duels_won_pct', 'tackles_won', 'key_passes', 'dribbles_completed', 'yellow_cards', 'red_cards', 'distance_covered_km', 'sprint_speed_kmh', 'saves', 'save_pct', 'clean_sheets', 'market_value_eur', 'contract_expiry', 'scout_rating', 'injury_days_last_season', 'observation_date', 'scout_name'].


In [27]:
for msg in risposta4["messages"]:
    print(f"--- {msg.type} ---")
    print(msg.content)
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print("Tool calls:", msg.tool_calls)
    print()

--- human ---
Qual è lo stipendio medio degli scout?

--- ai ---
Il dataset non contiene informazioni sugli stipendi degli scout. Le colonne disponibili nel dataframe sono: ['player_id', 'player_name', 'club', 'nationality', 'date_of_birth', 'position', 'preferred_foot', 'height_cm', 'weight_kg', 'season', 'matches_played', 'minutes_played', 'goals', 'assists', 'xg', 'xa', 'pass_accuracy_pct', 'duels_won_pct', 'aerial_duels_won_pct', 'tackles_won', 'key_passes', 'dribbles_completed', 'yellow_cards', 'red_cards', 'distance_covered_km', 'sprint_speed_kmh', 'saves', 'save_pct', 'clean_sheets', 'market_value_eur', 'contract_expiry', 'scout_rating', 'injury_days_last_season', 'observation_date', 'scout_name'].



**Test 5 — colonna con missing-by-design e valore categoriale sbagliato.**
`save_pct` è popolata solo per i portieri. Vediamo se l'agente, dopo aver tentato un filtro con un
valore di `position` sbagliato (dataframe vuoto), si accorge dell'errore invece di concludere che
il dato non esiste nel dataset.

In [28]:
risposta5 = agent.invoke({
    "messages": [{"role": "user", "content": "Chi è il portiere con la percentuale di parate (save_pct) più alta, considerando l'osservazione più recente di ciascun giocatore?"}]
})

print(risposta5["messages"][-1].content)

Il portiere con la percentuale di parate più alta, considerando l'osservazione più recente, è **Danny Claes**. Ecco i dettagli:

- **Club**: Monte Grigio SC
- **Nazionalità**: Belgio
- **Data di nascita**: 22 Giugno 2003
- **Altezza**: 182 cm
- **Peso**: 90.8 kg
- **Stagione**: 2025/26
- **Partite giocate**: 23
- **Minuti giocati**: 1616
- **Goal subiti**: 0
- **Assist**: 0
- **Percentuale di parate (save_pct)**: 80.3%
- **Clean sheets**: 5
- **Valore di mercato**: 4,662,000 EUR
- **Data di osservazione**: 10 Marzo 2026
- **Scout rating**: 78

Se hai bisogno di ulteriori informazioni, fammi sapere!


In [29]:
for msg in risposta5["messages"]:
    print(f"--- {msg.type} ---")
    print(msg.content)
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print("Tool calls:", msg.tool_calls)
    print()

--- human ---
Chi è il portiere con la percentuale di parate (save_pct) più alta, considerando l'osservazione più recente di ciascun giocatore?

--- ai ---

Tool calls: [{'name': 'python_repl', 'args': {'code': "recent_observations = df.sort_values('observation_date').drop_duplicates(subset='player_id', keep='last')\ngoalkeepers = recent_observations[recent_observations['position'] == 'Goalkeeper']\nbest_goalkeeper = goalkeepers.loc[goalkeepers['save_pct'].idxmax()]\nbest_goalkeeper"}, 'id': 'call_tDhYiUw0vrFX6dRdvSLU2asc', 'type': 'tool_call'}]

--- tool ---
Errore nell'esecuzione: attempt to get argmax of an empty sequence

--- ai ---

Tool calls: [{'name': 'python_repl', 'args': {'code': "df['position'].unique()"}, 'id': 'call_F0SweLNOmW7if8shfNCRYeWL', 'type': 'tool_call'}]

--- tool ---
<StringArray>
['CM', 'CAM', 'ST', 'RM', 'CDM', 'GK', 'LW', 'LB', 'RW', 'LM', 'RB', 'CB']
Length: 12, dtype: str


--- ai ---

Tool calls: [{'name': 'python_repl', 'args': {'code': "recent_observati

**Test 6 — stato persistente tra chiamate (prima verifica, indiretta).**
Chiediamo all'agente di calcolare e "ricordare" una variabile in una chiamata, poi di riusarla in
una chiamata successiva e separata.

In [30]:
risposta6a = agent.invoke({
    "messages": [{"role": "user", "content": "Calcola l'osservazione più recente per ciascun giocatore e salvala in una variabile chiamata 'recent_players'."}]
})
print(risposta6a["messages"][-1].content)

L'osservazione più recente per ciascun giocatore è stata calcolata e salvata nella variabile chiamata 'recent_players'.


In [31]:
risposta6b = agent.invoke({
    "messages": [{"role": "user", "content": "Usando la variabile 'recent_players' che hai già calcolato, quanti giocatori italiani ci sono?"}]
})
print(risposta6b["messages"][-1].content)

Ci sono 3 giocatori italiani nel dataset.


In [32]:
for msg in risposta6b["messages"]:
    print(f"--- {msg.type} ---")
    print(msg.content)
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print("Tool calls:", msg.tool_calls)
    print()

--- human ---
Usando la variabile 'recent_players' che hai già calcolato, quanti giocatori italiani ci sono?

--- ai ---

Tool calls: [{'name': 'python_repl', 'args': {'code': "recent_players = df.sort_values('observation_date').drop_duplicates(subset='player_id', keep='last')\nitalian_players_count = recent_players[recent_players['nationality'] == 'Italy'].shape[0]\nitalian_players_count"}, 'id': 'call_az2PQxZL8AnU04hiKWKFdqsM', 'type': 'tool_call'}]

--- tool ---
3


--- ai ---
Ci sono 3 giocatori italiani nel dataset.



**Test 7 — stato persistente tra chiamate (verifica diretta e definitiva).**
Una variabile "sentinella" (`numero_magico`) viene definita in una chiamata e letta, senza essere
ridefinita, in una chiamata completamente separata: l'unico modo per riuscirci è che l'ambiente
Python del tool l'abbia davvero mantenuta in memoria tra le due invocazioni dell'agente.

In [33]:
risposta7a = agent.invoke({
    "messages": [{"role": "user", "content": "Esegui esattamente questo codice e nient'altro: numero_magico = 12345"}]
})
print(risposta7a["messages"][-1].content)

Il codice è stato eseguito con successo.


In [34]:
risposta7b = agent.invoke({
    "messages": [{"role": "user", "content": "Stampa il valore della variabile 'numero_magico', senza definirla o ricalcolarla tu stesso."}]
})
print(risposta7b["messages"][-1].content)

Il valore della variabile 'numero_magico' è 12345.


In [35]:
for msg in risposta7b["messages"]:
    print(f"--- {msg.type} ---")
    print(msg.content)
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print("Tool calls:", msg.tool_calls)
    print()

--- human ---
Stampa il valore della variabile 'numero_magico', senza definirla o ricalcolarla tu stesso.

--- ai ---

Tool calls: [{'name': 'python_repl', 'args': {'code': 'numero_magico'}, 'id': 'call_NrfgLATpKs0onbGZFRnx0coa', 'type': 'tool_call'}]

--- tool ---
12345


--- ai ---
Il valore della variabile 'numero_magico' è 12345.



## 7. Setup consolidato dell'agente

Raccogliamo tool, system prompt e creazione dell'agente in un'unica funzione riutilizzabile,
che richiameremo anche dal futuro microservizio FastAPI.

In [36]:
def build_data_agent(df, llm):
    """Costruisce un agente ReAct per l'analisi del dataframe 'df'.
    Restituisce l'agente pronto per .invoke()."""

    python_tool = make_python_tool(df)

    system_prompt_text = f"""Colonne disponibili nel dataframe 'df': {list(df.columns)}

Nota importante sulla struttura dei dati: ogni riga rappresenta una singola osservazione di scouting
in un momento nel tempo (colonna 'observation_date'). Lo stesso giocatore ('player_id') compare più
volte, e metriche come 'market_value_eur' o 'scout_rating' possono cambiare da un'osservazione
all'altra nel tempo.

Usa sempre 'player_id' come chiave univoca del giocatore (mai 'player_name', per rischio di omonimie).

Se la domanda riguarda lo stato ATTUALE di un giocatore (es. valore di mercato attuale, ultimo rating),
NON prendere il valore massimo/minimo storico. Usa esattamente questo procedimento per ottenere
la riga più recente per ciascun giocatore:
    df.sort_values('observation_date').drop_duplicates(subset='player_id', keep='last')
Poi applica eventuali filtri o ordinamenti richiesti dalla domanda su questo sottoinsieme.

Se una domanda richiede informazioni non presenti tra le colonne del dataframe, NON rispondere con
conoscenza generale esterna: dichiara esplicitamente che il dataset non contiene quell'informazione,
elencando le colonne disponibili più pertinenti alla domanda.

Prima di filtrare su una colonna categoriale (es. 'position', 'preferred_foot', 'nationality') con un
valore specifico di cui non sei certo, controlla SEMPRE prima i valori validi con df['colonna'].unique(),
invece di assumere un valore plausibile. Se un filtro produce un risultato vuoto, non concludere che il
dato non esiste nel dataset: verifica prima se stai usando il valore/etichetta corretta."""

    return create_agent(llm, [python_tool], system_prompt=system_prompt_text)


# test rapido: ricreiamo l'agente con la funzione consolidata e ripetiamo una domanda già vista
agent = build_data_agent(df_clean, llm)

risposta_test = agent.invoke({
    "messages": [{"role": "user", "content": "Qual è lo scout_rating medio per ruolo? Rispondi in modo discorsivo."}]
})
print(risposta_test["messages"][-1].content)

Analizzando lo scout_rating medio per ruolo, emergono alcuni dati interessanti. I giocatori che ricoprono il ruolo di **CAM** (centrocampista offensivo) hanno il punteggio medio più alto, con un valore di **69.36**. Questo suggerisce che i talenti in questa posizione sono generalmente valutati molto positivamente dagli scout.

Seguono i **portieri** (GK) con un punteggio medio di **68.29**, il che indica che anche i portieri di qualità sono ben rappresentati nel dataset. 

I **centrocampisti** (CM) e i **centrocampisti difensivi** (CDM) hanno punteggi medi rispettivamente di **65.40** e **63.70**, mostrando che anche in queste posizioni ci sono giocatori di buon livello.

Per quanto riguarda i difensori, i **centrali** (CB) hanno un punteggio medio di **62.17**, mentre i **terzini sinistri** (LB) e **terzini destri** (RB) si attestano su **65.53** e **59.67** rispettivamente, con i terzini destri che mostrano un punteggio medio più basso.

Infine, i **punta** (ST) e i **esterni** (LW, 

## 8. Revisione post-prototipo

Dopo il prototipo il lavoro è passato in `cleaning.py`, `agent.py` e `main.py`. Le celle seguenti importano i moduli veri e mostrano i numeri attuali. Cosa è cambiato rispetto alle sezioni 1-7, e perché:

- **Club normalizzati (74 → 25).** Il prototipo non normalizzava `club`, ma nel file ci sono 74 grafie per 25 club reali. Ora una mappatura canonica (`CLUB_CANONICI`) con confronto senza distinzione tra maiuscole e minuscole le riconduce a 25; un club sconosciuto blocca l'avvio con un errore esplicito.
- **Deduplica dopo le normalizzazioni.** Nel prototipo `drop_duplicates()` era il primo passo (1208 → 1195 righe). Normalizzando prima, altri **10 duplicati nascosti** diventano identici ed emergono: il dataset finale ha **1185 righe**.
- **Osservazioni nello stesso giorno.** Esistono 5 coppie con stesso `player_id` e stessa data (giocatori 10, 33, 60, 70, 74). Sono mantenute e documentate; lo stato "attuale" è l'ultima riga registrata, con ordinamento stabile.
- **Report delle correzioni.** `clean_scouting_data` restituisce ora `(df, report)`. Il testo del report entra nel system prompt, perché i numeri delle correzioni non si possono ricostruire dal dataframe già pulito.
- **Agente costruito a ogni richiesta.** Nel prototipo un unico agente teneva lo stato del REPL tra una domanda e l'altra, quindi un filtro su `df` "contagiava" le richieste successive. Ora ogni richiesta costruisce un agente con il proprio namespace e una copia di `df`; la firma è `build_data_agent(df, llm, cleaning_summary)`.
- **Grafici.** La cartella dei PNG è un percorso assoluto, e al modello torna solo il nome del file, mai un percorso del disco.

I moduli sono importati con un prefisso o un alias (`cleaning.…`, `agent_module.…`) per non sovrascrivere le funzioni del prototipo, e in particolare la variabile `agent` già presente nel notebook.

La sezione 8 riusa `llm` e la chiave API definite nelle sezioni 1 e 6.2: va eseguita dopo di esse (o con *Run All*).

In [37]:
import pandas as pd
import cleaning
import agent as agent_module  # alias: nel notebook esiste già una variabile 'agent'

# Nomi con suffisso _rev: non sovrascrivono df, df_clean e le funzioni del prototipo
df_raw_rev = pd.read_csv("../data/scouting_data.csv")
df_rev, report_rev = cleaning.clean_scouting_data(df_raw_rev)

print("Righe: file grezzo", len(df_raw_rev), "-> dataset pulito", len(df_rev))
print("Grafie di club:", report_rev['club']['grafie_originali'], "->", report_rev['club']['grafie_finali'])
print("Duplicati:", report_rev['duplicati'])
print("Osservazioni nello stesso giorno:", report_rev['osservazioni_stesso_giorno'])
print("Giocatori unici:", report_rev['giocatori_unici'])
print()
print(cleaning.format_cleaning_report(report_rev))

Righe: file grezzo 1208 -> dataset pulito 1185
Grafie di club: 74 -> 25
Duplicati: {'rimossi_totale': 23, 'identici_nel_file_grezzo': 13, 'emersi_dopo_le_normalizzazioni': 10}
Osservazioni nello stesso giorno: {'coppie': 5, 'righe_coinvolte': 10, 'player_id': [10, 33, 60, 70, 74]}
Giocatori unici: 130

Correzioni applicate ai dati grezzi (file originale: 1208 righe; dataset pulito: 1185 righe, 130 giocatori).
I conteggi delle correzioni sono riferiti alle righe del file grezzo; quelli delle imputazioni alle righe rimaste dopo la rimozione dei duplicati.
- Duplicati rimossi: 23 righe (13 identiche già nel file grezzo, 10 diventate identiche solo dopo le normalizzazioni).
- Nazionalità: 45 grafie normalizzate in 24 (72 righe modificate).
- Club: 74 grafie normalizzate in 25 club (151 righe modificate).
- Valore di mercato: 120 righe in formato italiano (es. 11.415.000,00) convertite in numero.
- Date: 181 date di osservazione e 0 scadenze di contratto in formato GG/MM/AAAA convertite.
- 

In [38]:
# Nuova firma: build_data_agent(df, llm, cleaning_summary). Il riepilogo delle correzioni
# viene dal report di clean_scouting_data e finisce nel system prompt.
cleaning_summary_rev = cleaning.format_cleaning_report(report_rev)
agent_rev = agent_module.build_data_agent(df_rev, llm, cleaning_summary_rev)

risposta_rev = agent_rev.invoke({
    "messages": [{"role": "user", "content": "Qual è il valore di mercato attuale più alto nel dataset? Indica player_id, club e valore."}]
})
print(risposta_rev["messages"][-1].content)

Il valore di mercato attuale più alto nel dataset è di 18.680.000,00 EUR. I dettagli sono i seguenti:

- **player_id**: 5
- **club**: Fiume Rosso AC
- **valore di mercato**: 18.680.000,00 EUR


**Nota sugli output.** Gli output di questo notebook risalgono al 6 ottobre 2026. Le evoluzioni successive del data agent (codici di ruolo, controllo delle osservazioni multiple, tabelle complete, grafici salvati dal tool, titoli nei nomi, assi dei grafici, forme dell'id) sono descritte nel [README](../README.md) e in [`docs/TEST.md`](../docs/TEST.md).